# FactChecker AI - Transformer Fine-Tuning
1. Runtime > Change runtime type > T4 GPU
2. Add HF_TOKEN to Colab secrets (key icon, left sidebar)
3. Runtime > Run all

In [1]:
# Cell 1: Install compatible versions together
!pip install -q transformers==4.41.3 huggingface_hub==0.23.4 datasets evaluate accelerate scikit-learn pandas numpy
import transformers, torch, datasets, sklearn
print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')
print(f'datasets     : {datasets.__version__}')
print('PASS: install OK')

ERROR: Ignored the following yanked versions: 4.14.0, 4.25.0, 4.46.0, 4.52.0, 4.57.0
ERROR: Could not find a version that satisfies the requirement transformers==4.41.3 (from versions: 0.1, 2.0.0, 2.1.0, 2.1.1, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 3.0.0, 3.0.1, 3.0.2, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1, 4.0.0rc1, 4.0.0, 4.0.1, 4.1.0, 4.1.1, 4.2.0, 4.2.1, 4.2.2, 4.3.0rc1, 4.3.0, 4.3.1, 4.3.2, 4.3.3, 4.4.0, 4.4.1, 4.4.2, 4.5.0, 4.5.1, 4.6.0, 4.6.1, 4.7.0, 4.8.0, 4.8.1, 4.8.2, 4.9.0, 4.9.1, 4.9.2, 4.10.0, 4.10.1, 4.10.2, 4.10.3, 4.11.0, 4.11.1, 4.11.2, 4.11.3, 4.12.0, 4.12.1, 4.12.2, 4.12.3, 4.12.4, 4.12.5, 4.13.0, 4.14.1, 4.15.0, 4.16.0, 4.16.1, 4.16.2, 4.17.0, 4.18.0, 4.19.0, 4.19.1, 4.19.2, 4.19.3, 4.19.4, 4.20.0, 4.20.1, 4.21.0, 4.21.1, 4.21.2, 4.21.3, 4.22.0, 4.22.1, 4.22.2, 4.23.0, 4.23.1, 4.24.0, 4.25.1, 4.26.0, 4.26.1, 4.27.0, 4.27.1, 4.27.2, 4.27.3, 4.27.4, 4.28.0, 4.28.1, 4.29.0, 4.29.1, 4.29.2, 4

In [2]:
# Cell 2: Auth + GPU
import torch
from huggingface_hub import login

HF_TOKEN = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    login(token=tok)
    HF_TOKEN = tok
    print('HuggingFace: LOGGED IN')
except Exception as e:
    print(f'HuggingFace: {e}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB')
else:
    print('WARN: No GPU!')

HF_USERNAME = 'Bharat2004'
MODEL_NAME  = f'{HF_USERNAME}/factchecker-deberta'
BASE_MODEL  = 'distilbert-base-uncased'
print(f'Push to: {MODEL_NAME}')

HuggingFace: LOGGED IN
GPU: Tesla T4 | 14 GB
Push to: Bharat2004/factchecker-deberta


In [3]:
# Cell 3: Load data
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print('Loading GonzaloA/fake_news...')
ds = load_dataset('GonzaloA/fake_news', split='train')
df = ds.to_pandas()
title = df['title'].fillna('') if 'title' in df.columns else pd.Series(['']*len(df))
df['text']  = (title + ' ' + df['text'].fillna('')).str.strip()
df['label'] = df['label'].astype(int)
print(f'Loaded: {len(df)} rows | {df["label"].value_counts().to_dict()}')

fake   = df[df['label']==1]
real   = df[df['label']==0]
target = min(11000, min(len(fake), len(real)))
df_bal = pd.concat([
    fake.sample(target, random_state=42),
    real.sample(target, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp  = train_test_split(df_bal, test_size=0.15, random_state=42, stratify=df_bal['label'])
val_df, test_df = train_test_split(temp,   test_size=0.50, random_state=42, stratify=temp['label'])

print(f'Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')
print(f'Balance: {train_df["label"].value_counts().to_dict()}')
print('PASS: data ready')

Loading GonzaloA/fake_news...


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/38.8M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Loaded: 24353 rows | {1: 13195, 0: 11158}
Train:18700 Val:1650 Test:1650
Balance: {0: 9350, 1: 9350}
PASS: data ready


In [4]:
# Cell 4: Tokenize
from transformers import AutoTokenizer
from datasets import Dataset

MAX_LEN   = 256
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

def to_ds(df):
    d = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label'].tolist()})
    return d.map(tokenize, batched=True, batch_size=256, remove_columns=['text'])

print('Tokenizing...')
train_ds = to_ds(train_df)
val_ds   = to_ds(val_df)
test_ds  = to_ds(test_df)
train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

s = train_ds[0]
assert len(s['input_ids']) == MAX_LEN
assert s['labels'].item() in [0, 1]
print(f'Token length: {len(s["input_ids"])} | Label: {s["labels"].item()}')
print('PASS: tokenization OK')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing...


Map:   0%|          | 0/18700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1650 [00:00<?, ? examples/s]

Map:   0%|          | 0/1650 [00:00<?, ? examples/s]

Token length: 256 | Label: 0
PASS: tokenization OK


In [5]:
# Cell 5: Model + trainer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'REAL', 1:'FAKE'},
    label2id={'REAL':0, 'FAKE':1}
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Forward pass check
model.eval()
with torch.no_grad():
    keys   = [k for k in ['input_ids','attention_mask'] if k in train_ds[0]]
    sample = {k: train_ds[0][k].unsqueeze(0).to(device) for k in keys}
    out    = model(**sample)
    assert out.logits.shape == (1, 2)
    assert not torch.isnan(out.logits).any(), 'NaN in logits!'
    print(f'Forward pass OK: {out.logits.cpu().numpy()}')
model.train()

acc_m = evaluate.load('accuracy')
f1_m  = evaluate.load('f1')

def compute_metrics(ep):
    preds = ep[0].argmax(axis=-1)
    return {
        'accuracy': acc_m.compute(predictions=preds, references=ep[1])['accuracy'],
        'f1_macro': f1_m.compute(predictions=preds, references=ep[1], average='macro')['f1']
    }

args = TrainingArguments(
    output_dir                  = './factchecker-model',
    num_train_epochs            = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_steps                = 100,
    lr_scheduler_type           = 'linear',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    fp16                        = (device == 'cuda'),
    bf16                        = False,
    gradient_accumulation_steps = 2,
    max_grad_norm               = 1.0,
    dataloader_num_workers      = 0,
    logging_steps               = 50,
    report_to                   = 'none',
    save_total_limit            = 1,
    seed                        = 42,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print('PASS: trainer ready')

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
# Cell 6: Train
print('Training (~20 min on T4)...')
result = trainer.train()
print(f'Steps:{result.global_step} Loss:{result.training_loss:.4f}')
assert result.training_loss > 0.01
print('PASS: training complete')

In [ ]:
# Cell 7: Evaluate
from sklearn.metrics import classification_report, brier_score_loss
import torch.nn.functional as F
import torch, json, datetime, os, numpy as np
from transformers import pipeline as hf_pipeline

res    = trainer.evaluate(test_ds)
acc    = res['eval_accuracy']
f1     = res['eval_f1_macro']
print(f'Accuracy: {acc:.4f}  F1: {f1:.4f}')

out        = trainer.predict(test_ds)
y_pred     = out.predictions.argmax(axis=-1)
y_true     = out.label_ids
print(classification_report(y_true, y_pred, target_names=['Real','Fake']))

probs      = F.softmax(torch.tensor(out.predictions.astype(np.float32)), dim=-1).numpy()
fake_probs = np.nan_to_num(probs[:,1], nan=0.5)
brier      = brier_score_loss(y_true, fake_probs)
print(f'Brier: {brier:.4f}')

clf = hf_pipeline('text-classification', model=model, tokenizer=tokenizer,
                  device=0 if device=='cuda' else -1)
tests = [
    ('SHOCKING: Government putting microchips in vaccines!!!', 'FAKE'),
    ('Scientists discover water on Mars in new NASA study', 'REAL'),
    ('EXPOSED: They are hiding the cure for cancer!!!', 'FAKE'),
    ('Stock market closes higher after Fed announcement', 'REAL'),
]
passed = 0
for text, expected in tests:
    r = clf(text[:256])[0]
    ok = r['label'] == expected
    if ok: passed += 1
    print(f'  [{"PASS" if ok else "FAIL"}] {r["label"]} ({r["score"]:.2f}) | {text[:55]}')
print(f'Sanity: {passed}/{len(tests)}')

assert acc > 0.6, f'Accuracy too low: {acc}'
print('PASS: evaluation OK')

os.makedirs('./factchecker-model', exist_ok=True)
version = {
    'model': MODEL_NAME, 'base': BASE_MODEL,
    'version': datetime.datetime.utcnow().strftime('%Y%m%d_%H%M'),
    'accuracy': round(float(acc),4), 'f1_macro': round(float(f1),4),
    'brier_score': round(float(brier),4), 'train_samples': len(train_df)
}
with open('./factchecker-model/model_version.json','w') as f:
    json.dump(version, f, indent=2)
print(json.dumps(version, indent=2))

In [ ]:
# Cell 8: Push + download
import os
if HF_TOKEN:
    print(f'Pushing to {MODEL_NAME}...')
    trainer.push_to_hub(MODEL_NAME)
    tokenizer.push_to_hub(MODEL_NAME)
    print(f'Live: https://huggingface.co/{MODEL_NAME}')
else:
    trainer.save_model('./factchecker-model')
    tokenizer.save_pretrained('./factchecker-model')
    print('Saved locally')

from google.colab import files
files.download('./factchecker-model/model_version.json')
print(f'Done. Set Render env: DEBERTA_MODEL={MODEL_NAME}')